In [42]:
from langchain_groq import ChatGroq
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_community.tools import QuerySQLDataBaseTool
from langchain_core .prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.agents import create_agent
from langchain_community.utilities import SQLDatabase
from langchain_classic.chains import create_sql_query_chain
from sqlalchemy import create_engine
from operator import itemgetter
from pprint import pprint
import os

In [43]:
sql_agent_llm = ChatGroq(model="openai/gpt-oss-120b")
llm = HuggingFaceEndpoint(
    model = "google/gemma-2-2b-it",
    task="text-generation"
)

table_extractor_llm = ChatHuggingFace(llm = llm)

In [44]:
POSTGRES_HOST = os.getenv("POSTGRES_HOST")
POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_PASSWORD = os.getenv("POSTGRES_PASSWORD")
POSTGRES_PORT = os.getenv("POSTGRES_PORT")
POSTGRES_DATABASE = os.getenv("POSTGRES_DATABASE")

connection_string = (
    f"postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}"
    f"@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DATABASE}"
)

engine = create_engine(connection_string)

db = SQLDatabase(engine=engine)

print(db.dialect)
print(db.get_usable_table_names())

db.run("SELECT * FROM project;")


postgresql
['manager', 'project']


"[(3, 'Procurement Management', None, None, None, None), (4, None, None, None, None, None), (5, 'HIMS', None, None, None, None), (2, 'Payroll Management', 'Test Description', None, None, 'ON_HOLD'), (1, 'HR Management', 'Hr management Demo', datetime.date(2026, 1, 31), datetime.date(2026, 1, 3), 'ON_HOLD')]"

In [45]:
from pydantic import BaseModel, Field
from typing import List

class Table(BaseModel):
    name: str = Field(description="Name of a SQL table")

class TableList(BaseModel):
    tables: List[Table]

In [46]:
table_names = "\n".join(db.get_usable_table_names())
pprint(table_names)

'manager\nproject'


In [47]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = PromptTemplate(
    template="""
You are a database expert.

Available SQL tables:
{table_names}

Question:
{question}

Return ONLY valid JSON in this EXACT format:

{{
  "tables": ["table1", "table2"]
}}

Rules:
- Do NOT return schema
- Do NOT explain
- Do NOT add text
""",
    input_variables=["question", "table_names"],
)


In [48]:
parser = StrOutputParser()
table_chain = prompt | table_extractor_llm | parser


In [56]:
import json
import re

def parse_tables(output: str) -> list[str]:
    if not output or not output.strip():
        return []

    # 1️⃣ Try direct JSON
    try:
        data = json.loads(output)
        return data.get("tables", [])
    except Exception:
        pass

    # 2️⃣ Try to extract JSON object from text
    try:
        match = re.search(r"\{.*\}", output, re.DOTALL)
        if match:
            data = json.loads(match.group())
            return data.get("tables", [])
    except Exception:
        pass

    # 3️⃣ Fallback: comma / newline separated tables
    return [
        t.strip()
        for t in re.split(r"[,\n]", output)
        if t.strip()
    ]


In [62]:
raw = table_chain.invoke({
    "question": "?",
    "table_names": table_names
})

tables = parse_tables(raw)
print(tables)


['manager', 'project']
